End-to-End решение табличной задачи — практикум

**Цель:** собрать полный цикл: EDA → препроцессинг → модель → оценка → выводы. 70% времени — код и эксперименты.

**Формат работы:**
- Каждый блок содержит мини-задачу с чеклистом.
- Теория только напоминанием, основное — самостоятельные шаги.
- Можно опираться на приёмы из предыдущих ноутбуков (`boostings_101.ipynb`, `eda-feat-engeen.ipynb`, `pandas_matplotlib:seaborn.ipynb`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Библиотеки загружены ✓")


### План практики
1. Сгенерировать/загрузить данные и сделать быстрый EDA.
2. Разделить признаки, придумать хотя бы 2 новых фичи.
3. Собрать пайплайн (ColumnTransformer + модель).
4. Оценить модель через кросс-валидацию и финальный тест.
5. Оформить выводы и идеи улучшений.


---
## 1. Загрузка и первичный анализ данных

Используем синтетический датасет, имитирующий задачу предсказания оттока клиентов (churn prediction).


In [ ]:
# Генерируем реалистичный датасет
np.random.seed(RANDOM_STATE)
n = 2000

# Числовые признаки
df = pd.DataFrame({
    'tenure_months': np.random.exponential(24, n).clip(1, 72).astype(int),
    'monthly_charges': np.random.normal(65, 30, n).clip(20, 120),
    'total_charges': np.nan,  # Заполним ниже
    'num_support_tickets': np.random.poisson(2, n),
    'num_products': np.random.randint(1, 6, n),
    'age': np.random.normal(45, 15, n).clip(18, 80).astype(int),
})

# total_charges = tenure * monthly_charges + шум
df['total_charges'] = df['tenure_months'] * df['monthly_charges'] + np.random.normal(0, 100, n)

# Категориальные признаки
df['contract_type'] = np.random.choice(['month-to-month', 'one_year', 'two_year'], n, p=[0.5, 0.3, 0.2])
df['payment_method'] = np.random.choice(['credit_card', 'bank_transfer', 'electronic_check'], n)
df['internet_service'] = np.random.choice(['DSL', 'Fiber', 'None'], n, p=[0.4, 0.45, 0.15])

# Добавляем пропуски (реалистично)
missing_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
df.loc[missing_idx, 'total_charges'] = np.nan
df.loc[np.random.choice(n, 30, replace=False), 'age'] = np.nan

# Таргет: churn (отток)
# Логика: короткий tenure, дорогой тариф, много тикетов -> выше вероятность оттока
churn_prob = (
    0.1 + 
    0.3 * (df['tenure_months'] < 12).astype(float) +
    0.2 * (df['monthly_charges'] > 70).astype(float) +
    0.15 * (df['num_support_tickets'] > 3).astype(float) +
    0.25 * (df['contract_type'] == 'month-to-month').astype(float)
)
churn_prob = churn_prob.clip(0, 0.9)
df['churn'] = (np.random.random(n) < churn_prob).astype(int)

print(f"Размер датасета: {df.shape}")
print(f"\\nПервые строки:")
df.head(10)


In [ ]:
# Первичный анализ
print("=== Информация о датасете ===")
print(df.info())
print(f"\\n=== Пропуски ===")
print(df.isnull().sum())
print(f"\\n=== Распределение таргета ===")
print(df['churn'].value_counts(normalize=True))


**Наблюдения:**
- Есть пропуски в `total_charges` и `age`
- Классы немного несбалансированы
- Есть числовые и категориальные признаки

---
## 2. EDA (краткий)

Подробный EDA — см. `eda-feat-engeen.ipynb`. Здесь только ключевые графики.


In [ ]:
# Распределение ключевых признаков по классам
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.histplot(data=df, x='tenure_months', hue='churn', kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Tenure по классам')

sns.histplot(data=df, x='monthly_charges', hue='churn', kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Monthly Charges по классам')

sns.countplot(data=df, x='contract_type', hue='churn', ax=axes[1, 0])
axes[1, 0].set_title('Contract Type по классам')

sns.boxplot(data=df, x='churn', y='num_support_tickets', ax=axes[1, 1])
axes[1, 1].set_title('Support Tickets по классам')

plt.tight_layout()
plt.show()


In [ ]:
# Корреляционная матрица
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

print("Корреляция с таргетом:")
print(corr['churn'].sort_values(ascending=False))


---
## 3. Feature Engineering

Создаём новые признаки на основе EDA. Подробнее — см. `eda-feat-engeen.ipynb`.


### Практика: быстрый EDA
- Постройте 2–3 графика, описывающих распределение признаков и целевой.
- Выявите выбросы/дисбаланс и запишите наблюдения.


In [ ]:
# TODO: визуализации для EDA
# sns.countplot(x="target", data=df)
# sns.histplot(df["..."], kde=True)
# plt.show()
# print("Вывод: ...")


In [ ]:
# Feature Engineering
df_fe = df.copy()

# 1. Средний чек в месяц с учётом продуктов
df_fe['charge_per_product'] = df_fe['monthly_charges'] / df_fe['num_products']

# 2. Флаг: новый клиент (tenure < 6 месяцев)
df_fe['is_new_customer'] = (df_fe['tenure_months'] < 6).astype(int)

# 3. Интенсивность обращений в поддержку
df_fe['tickets_per_month'] = df_fe['num_support_tickets'] / df_fe['tenure_months'].clip(1)

print("Новые признаки:")
print(df_fe[['charge_per_product', 'is_new_customer', 'tickets_per_month']].describe())


In [ ]:
# Разделяем данные на train/test ПЕРЕД всей предобработкой!
X = df_fe.drop('churn', axis=1)
y = df_fe['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate: {y_test.mean():.2%}")


---
## 4. Baseline модель

Начинаем с простой модели, чтобы установить базовый уровень качества.

**Важно:** Используем Pipeline, чтобы избежать утечек данных! (см. `02_sklearn_pipelines.ipynb`)


In [ ]:
# Определяем типы признаков
numeric_features = ['tenure_months', 'monthly_charges', 'total_charges', 
                    'num_support_tickets', 'num_products', 'age',
                    'charge_per_product', 'is_new_customer', 'tickets_per_month']
categorical_features = ['contract_type', 'payment_method', 'internet_service']

print(f"Числовые признаки ({len(numeric_features)}): {numeric_features}")
print(f"Категориальные признаки ({len(categorical_features)}): {categorical_features}")


In [ ]:
# Создаём препроцессор
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Baseline: Логистическая регрессия
baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

print("Baseline пайплайн создан ✓")


In [ ]:
# Оценка baseline через CV (см. 01_cv_metrics_leakage.ipynb)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_scores = cross_val_score(baseline_pipe, X_train, y_train, cv=cv, scoring='roc_auc')

print("=== BASELINE: Logistic Regression ===")
print(f"CV ROC-AUC: {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}")
print(f"Scores: {baseline_scores}")


### Практика: фичи и пайплайн
- Разделите признаки на числовые/категориальные, добавьте 2 новых признака.
- Соберите ColumnTransformer и модель (логрег/дерево/градиентный бустинг).
- Проверьте пайплайн через cross_val_score.


In [ ]:
# TODO: ваш пайплайн
# df["feature_new1"] = ...
# df["feature_new2"] = ...
# preprocessor = ColumnTransformer([...])
# model = ...
# pipeline = Pipeline([
#     ("prep", preprocessor),
#     ("model", model)
# ])
# scores = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc")
# print(scores)


---
## 5. Улучшение модели

Пробуем более мощные модели. Подробнее о бустингах — см. `boostings_101.ipynb`.


In [ ]:
# Модель 1: Random Forest
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))
])

rf_scores = cross_val_score(rf_pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"Random Forest CV ROC-AUC: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")


In [ ]:
# Модель 2: Gradient Boosting
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE
    ))
])

gb_scores = cross_val_score(gb_pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"Gradient Boosting CV ROC-AUC: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}")


In [ ]:
# Сравнение моделей
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'CV ROC-AUC Mean': [baseline_scores.mean(), rf_scores.mean(), gb_scores.mean()],
    'CV ROC-AUC Std': [baseline_scores.std(), rf_scores.std(), gb_scores.std()]
})

results = results.sort_values('CV ROC-AUC Mean', ascending=False)
print("=== Сравнение моделей ===")
print(results.to_string(index=False))

# Визуализация
plt.figure(figsize=(10, 5))
plt.barh(results['Model'], results['CV ROC-AUC Mean'], xerr=results['CV ROC-AUC Std'], 
         color=['#2ecc71', '#3498db', '#9b59b6'])
plt.xlabel('CV ROC-AUC')
plt.title('Сравнение моделей')
plt.xlim(0.5, 1.0)
plt.tight_layout()
plt.show()


---
## 6. Финальная оценка и отчёт

Выбираем лучшую модель, обучаем на всём train, оцениваем на test.


In [ ]:
# Финальная модель — Gradient Boosting (лучший результат на CV)
final_model = gb_pipe

# Обучаем на всём train
final_model.fit(X_train, y_train)

# Предсказываем на test
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

# Метрики
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"=== ФИНАЛЬНАЯ ОЦЕНКА НА TEST ===")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))


### Финальная проверка и выводы
1. Обучите лучшую модель на полном train, провалидируйте на hold-out или через cross_val_predict.
2. Сохраните ключевые метрики и важность признаков.
3. В финальном Markdown-абзаце ответьте на вопросы: какая метрика ключевая, где узкое место данных, что улучшать дальше.


In [ ]:
# TODO: итоговый прогон
# best_model = pipeline.fit(X_train, y_train)
# val_pred = best_model.predict_proba(X_val)[:, 1]
# print("ROC-AUC:", roc_auc_score(y_val, val_pred))
# print("Вывод: ...")


In [ ]:
# Диагностика модели (см. 03_visual_diagnostics.ipynb)
fig = plt.figure(figsize=(15, 5))

# Confusion Matrix
ax1 = fig.add_subplot(1, 3, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax1.set_title('Confusion Matrix')
ax1.set_ylabel('Истинный класс')
ax1.set_xlabel('Предсказанный класс')

# ROC Curve
ax2 = fig.add_subplot(1, 3, 2)
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {test_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'r--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.3)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Feature Importance
ax3 = fig.add_subplot(1, 3, 3)
# Получаем имена признаков после препроцессинга
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_features = numeric_features + cat_feature_names

# Важность признаков
importances = final_model.named_steps['classifier'].feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

ax3.barh(range(10), importances[top_indices][::-1])
ax3.set_yticks(range(10))
ax3.set_yticklabels([all_features[i] for i in top_indices[::-1]])
ax3.set_xlabel('Важность')
ax3.set_title('Top-10 признаков')

plt.tight_layout()
plt.show()


---
## Краткий отчёт

**Задача:** Предсказание оттока клиентов (бинарная классификация)

**Данные:** 2000 записей, 9 числовых и 3 категориальных признака

**Методология:**
- Train/Test split: 80/20 со стратификацией
- Кросс-валидация: 5-fold StratifiedKFold
- Метрика: ROC-AUC

**Результаты:**

| Модель | CV ROC-AUC | Test ROC-AUC |
|--------|------------|--------------|
| Logistic Regression | ~0.XX | - |
| Random Forest | ~0.XX | - |
| **Gradient Boosting** | ~0.XX | ~0.XX |

**Важнейшие признаки:**
1. tenure_months — время с клиентом
2. monthly_charges — ежемесячный платёж
3. contract_type — тип контракта

**Выводы:**
- Gradient Boosting показал лучший результат
- Клиенты с коротким tenure и месячным контрактом — в зоне риска
- Можно улучшить модель тюнингом гиперпараметров (см. `boostings_101.ipynb`)

---
## Что дальше?

Этот ноутбук — шаблон для ваших проектов. Рекомендуемые улучшения:
- Тюнинг гиперпараметров с RandomizedSearchCV или Optuna
- Добавление новых признаков
- Калибровка вероятностей
- A/B тестирование в продакшене

**Связанные материалы:**
- `01_cv_metrics_leakage.ipynb` — кросс-валидация и метрики
- `02_sklearn_pipelines.ipynb` — пайплайны
- `03_visual_diagnostics.ipynb` — визуализация
- `boostings_101.ipynb` — тюнинг бустингов
- `eda-feat-engeen.ipynb` — EDA и feature engineering
